# Export Formats

The `exports` module converts flight plans into file formats used by
flight crews, data archives, and briefing workflows. All formats are
drop-in compatible with [MovingLines](https://github.com/samuelleblanc/fp),
NASA's standard flight planning tool.

This notebook covers:

1. Building a sample flight plan
2. Coordinate formatting and magnetic declination
3. Extracting a waypoint table
4. Pilot Excel export
5. ForeFlight CSV for iPad
6. Honeywell FMS for G-III/G-IV avionics
7. ER-2 CSV
8. ICARTT for NASA data archiving
9. KML/KMZ for Google Earth
10. GPX for GPS devices
11. Plain text
12. Full working Excel (round-trip with MovingLines)

In [1]:
import datetime
import os
import tempfile

import pandas as pd

from hyplan import (
    KingAirB200, NASA_ER2,
    Airport, initialize_data,
    compute_flight_plan,
    FlightLine,
)
from hyplan.waypoint import Waypoint
from hyplan.units import ureg
from hyplan.geometry import (
    magnetic_declination, true_to_magnetic,
    dd_to_ddm, dd_to_ddms, dd_to_nddmm, dd_to_foreflight_oneline,
)
from hyplan.exports import (
    extract_waypoints, generate_wp_names,
    to_excel, to_pilot_excel,
    to_foreflight_csv, to_honeywell_fms, to_er2_csv,
    to_icartt, to_kml, to_gpx, to_txt,
)

# Output directory for generated files
OUT_DIR = tempfile.mkdtemp(prefix="hyplan_exports_")
print(f"Export files will be written to: {OUT_DIR}")

Export files will be written to: /var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/hyplan_exports_naxevlsj


## 1. Build a Sample Flight Plan

We create a multi-waypoint flight plan from Edwards AFB using a
King Air B200, with altitude changes between waypoints — a realistic
scenario that exercises all export features.

In [2]:
initialize_data(countries=["US"])
b200 = KingAirB200()
kedw = Airport("KEDW")

waypoints = [
    Waypoint(34.7, -118.2, 0.0,
             altitude_msl=ureg.Quantity(10000, "feet"), name="WP1"),
    Waypoint(34.9, -118.0, 45.0,
             altitude_msl=ureg.Quantity(20000, "feet"), name="WP2"),
    Waypoint(35.1, -118.2, 180.0,
             altitude_msl=ureg.Quantity(15000, "feet"), name="WP3"),
    Waypoint(34.8, -118.4, 270.0,
             altitude_msl=ureg.Quantity(18000, "feet"), name="WP4"),
]

plan = compute_flight_plan(
    b200, waypoints,
    takeoff_airport=kedw, return_airport=kedw,
)

# Mission metadata
TAKEOFF_TIME = datetime.datetime(2025, 6, 15, 15, 0, 0)
MISSION_NAME = "HYPLAN-DEMO"

print(f"Flight plan: {len(plan)} segments")
print(f"Aircraft: {b200.aircraft_type} ({b200.tail_number})")
print(f"Airport: {kedw.icao_code} — {kedw.name}")
print()
plan[["segment_type", "segment_name", "distance", "time_to_segment"]]

Flight plan: 5 segments
Aircraft: King Air 200 (Unknown)
Airport: KEDW — Edwards Air Force Base



,segment_type,segment_name,distance,time_to_segment
0,takeoff,Departure,21.628278,6.374650
1,climb,WP1 to WP2,18.696180,5.049772
2,descent,WP2 to WP3,18.208500,5.132597
3,climb,WP3 to WP4,20.757736,5.701929
4,descent,Return,33.204964,10.524902


## 2. Coordinate Formatting and Magnetic Declination

Pilot exports need coordinates in various formats and magnetic headings.
These utilities live in `hyplan.geometry`.

In [3]:
# Example point: Edwards AFB
lat, lon = kedw.latitude, kedw.longitude
print(f"Decimal degrees:   {lat:.6f}, {lon:.6f}")
print(f"DD MM (pilot):     {dd_to_ddm(lat, lon)}")
print(f"DD MM SS:          {dd_to_ddms(lat, lon)}")
print(f"NDDD MM.SS (FMS):  {dd_to_nddmm(lat, lon)}")
print(f"ForeFlight inline: {dd_to_foreflight_oneline(lat, lon)}")
print()

# Magnetic declination
dec = magnetic_declination(lat, lon)
print(f"Magnetic declination at KEDW: {dec:.2f}° (positive = east)")
print(f"True heading 090° → Magnetic: {true_to_magnetic(90.0, dec):.1f}°")
print(f"True heading 360° → Magnetic: {true_to_magnetic(360.0, dec):.1f}°")

Decimal degrees:   34.910781, -117.886391
DD MM (pilot):     ('34 54.65', '-117 53.18')
DD MM SS:          ('34 54 38.8', '-117 53 11.0')
NDDD MM.SS (FMS):  ('N34 54.65', 'W117 53.18')
ForeFlight inline: N3454.647/W11753.183

Magnetic declination at KEDW: 11.33° (positive = east)
True heading 090° → Magnetic: 78.7°
True heading 360° → Magnetic: 348.7°


## 3. Waypoint Table Extraction

`extract_waypoints()` converts the segment-based flight plan into a
waypoint-based table — one row per waypoint with cumulative distance,
time, altitude, heading, and speed.

In [4]:
wps = extract_waypoints(plan)
print(f"{len(wps)} waypoints extracted from {len(plan)} segments\n")

display_cols = [
    "wp", "lat", "lon", "alt_kft", "heading",
    "speed_kt", "dist_nm", "cum_dist_nm",
    "leg_time_min", "cum_time_min", "segment_type",
]
wps[display_cols].round(2)

6 waypoints extracted from 5 segments



,wp,lat,lon,alt_kft,heading,speed_kt,dist_nm,cum_dist_nm,leg_time_min,cum_time_min,segment_type
0,0,34.91,-117.89,2.31,0.0,203.57,21.63,0.00,6.37,0.00,takeoff
1,1,34.70,-118.20,10.00,0.0,222.14,18.70,21.63,5.05,6.37,climb
2,2,34.90,-118.00,20.00,45.0,212.86,18.21,40.32,5.13,11.42,descent
3,3,35.10,-118.20,15.00,180.0,218.43,20.76,58.53,5.70,16.56,climb
4,4,34.80,-118.40,18.00,270.0,189.29,33.20,79.29,10.52,22.26,descent
5,5,34.91,-117.89,2.31,0.0,0.00,0.00,112.50,0.00,32.78,


In [5]:
# Waypoint naming (MovingLines-compatible 5-char names)
names = generate_wp_names(len(wps), prefix="B", date=TAKEOFF_TIME.date())
print("Generated waypoint names:")
for name, (_, wp) in zip(names, wps.iterrows()):
    print(f"  {name}  ({wp['lat']:.4f}, {wp['lon']:.4f})  {wp['alt_kft']:.1f} kft")

Generated waypoint names:
  B1500  (34.9108, -117.8864)  2.3 kft
  B1501  (34.7000, -118.2000)  10.0 kft
  B1502  (34.9000, -118.0000)  20.0 kft
  B1503  (35.1000, -118.2000)  15.0 kft
  B1504  (34.8000, -118.4000)  18.0 kft
  B1505  (34.9108, -117.8864)  2.3 kft


## 4. Pilot Excel

Simplified waypoint table for the flight crew, matching the
MovingLines `_for_pilots.xlsx` format. Includes waypoint names,
formatted coordinates, altitude, UTC time, and magnetic heading.

Three coordinate formats are supported via the `coord_format` parameter:
- `'DD MM'` — degrees and decimal minutes (default)
- `'DD MM SS'` — degrees, minutes, seconds
- `'NDDD MM.SS'` — hemisphere prefix with decimal minutes

In [6]:
pilot_path = os.path.join(OUT_DIR, "demo_for_pilots.xlsx")
to_pilot_excel(
    plan, pilot_path,
    aircraft=b200,
    takeoff_time=TAKEOFF_TIME,
    mission_name=MISSION_NAME,
    coord_format="DD MM",
    include_mag_heading=True,
)
print(f"Pilot Excel: {pilot_path}")
print(f"File size: {os.path.getsize(pilot_path):,} bytes")

# Read it back to verify
df = pd.read_excel(pilot_path, skiprows=1)
df

Pilot Excel: /var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/hyplan_exports_naxevlsj/demo_for_pilots.xlsx
File size: 6,409 bytes


,WP,WP name,Lat,Lon,Altitude [kft],UTC [hh:mm],Mag Heading [deg],Comments,Unnamed: 8,2025-06-15,HYPLAN-DEMO,Created with,hyplan
0,0,H1500,34 54.65,-117 53.18,2.312,15:00:00,348.669788,takeoff,NaN,NaN,NaN,NaN,NaN
1,1,H1501,34 42.00,-118 12.00,10.000,15:06:22.479000,348.640267,climb,NaN,NaN,NaN,NaN,NaN
2,2,H1502,34 54.00,-118 00.00,20.000,15:11:25.465000,33.647259,descent,NaN,NaN,NaN,NaN,NaN
3,3,H1503,35 06.00,-118 12.00,15.000,15:16:33.421000,168.569332,climb,NaN,NaN,NaN,NaN,NaN
4,4,H1504,34 48.00,-118 24.00,18.000,15:22:15.537000,258.580833,descent,NaN,NaN,NaN,NaN,NaN
5,5,H1505,34 54.65,-117 53.18,2.312,15:32:47.031000,348.669788,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,One line waypoints for foreflight:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,N3454.647/W11753.183 N3442.000/W11812.000 N345...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. ForeFlight CSV

Simple CSV for importing waypoints into the ForeFlight iPad app.
Also generates a companion `_oneline.txt` with all waypoints in
the compact `N3424.210/W11803.450` format for quick entry.

In [7]:
ff_path = os.path.join(OUT_DIR, "demo_FOREFLIGHT.csv")
to_foreflight_csv(plan, ff_path, takeoff_time=TAKEOFF_TIME)

print("=== ForeFlight CSV ===")
with open(ff_path) as f:
    print(f.read())

print("=== One-liner (for quick entry) ===")
with open(ff_path.replace(".csv", "_oneline.txt")) as f:
    print(f.read())

=== ForeFlight CSV ===
Waypoint,Description,LAT,LONG
H1500,ALT=2.31 kft takeoff,+34.910781000000,-117.886391000000
H1501,ALT=10.00 kft climb,+34.700000000000,-118.200000000000
H1502,ALT=20.00 kft descent,+34.900000000000,-118.000000000000
H1503,ALT=15.00 kft climb,+35.100000000000,-118.200000000000
H1504,ALT=18.00 kft descent,+34.800000000000,-118.400000000000
H1505,ALT=2.31 kft,+34.910780999889,-117.886391000492

=== One-liner (for quick entry) ===
N3454.647/W11753.183 N3442.000/W11812.000 N3454.000/W11800.000 N3506.000/W11812.000 N3448.000/W11824.000 N3454.647/W11753.183



## 6. Honeywell FMS

CSV format for Honeywell avionics on NASA G-III and G-IV aircraft.
Coordinates use the `NDD MM.SS` / `WDDD MM.SS` format.

In [8]:
hw_path = os.path.join(OUT_DIR, "demo_Honeywell.csv")
to_honeywell_fms(plan, hw_path, takeoff_time=TAKEOFF_TIME)

print("=== Honeywell FMS CSV ===")
with open(hw_path) as f:
    print(f.read())

=== Honeywell FMS CSV ===
E,WPT,FIX,LAT,LON
x,H1500,ALT=2.31 kft takeoff,N34 54.65,W117 53.18
x,H1501,ALT=10.00 kft climb,N34 42.00,W118 12.00
x,H1502,ALT=20.00 kft descent,N34 54.00,W118 00.00
x,H1503,ALT=15.00 kft climb,N35 06.00,W118 12.00
x,H1504,ALT=18.00 kft descent,N34 48.00,W118 24.00
x,H1505,ALT=2.31 kft,N34 54.65,W117 53.18



## 7. ER-2 CSV

Waypoint file for the NASA ER-2 high-altitude aircraft. Includes
all waypoints (no duplicate skipping) with UTC times.

In [9]:
er2_path = os.path.join(OUT_DIR, "demo_ER2.csv")
to_er2_csv(plan, er2_path, takeoff_time=TAKEOFF_TIME)

print("=== ER-2 CSV ===")
with open(er2_path) as f:
    print(f.read())

=== ER-2 CSV ===
ID,Description,LAT,LONG,Altitude [kft],UTC [hh:mm],Comments
 0,.H1500,+34.910781000000,-117.886391000000,2.31,15:00,takeoff
 1,.H1501,+34.700000000000,-118.200000000000,10.00,15:06,climb
 2,.H1502,+34.900000000000,-118.000000000000,20.00,15:11,descent
 3,.H1503,+35.100000000000,-118.200000000000,15.00,15:16,climb
 4,.H1504,+34.800000000000,-118.400000000000,18.00,15:22,descent
 5,.H1505,+34.910780999889,-117.886391000492,2.31,15:32,



## 8. ICARTT

NASA's standard format for airborne science data exchange (version 1001).
The flight plan is linearly interpolated at 60-second intervals to produce
a time series of lat, lon, altitude, speed, bearing, and solar zenith angle.

Filename convention: `{mission}-Flt-plan_{platform}_{YYYYMMDD}_{revision}.ict`

In [10]:
ict_path = os.path.join(OUT_DIR, f"{MISSION_NAME}-Flt-plan_B200_{TAKEOFF_TIME:%Y%m%d}_RA.ict")
to_icartt(
    plan, ict_path,
    pi_name="R. Pavlick",
    institution="NASA JPL",
    mission_name=MISSION_NAME,
    flight_date=TAKEOFF_TIME.date(),
    aircraft=b200,
    takeoff_time=TAKEOFF_TIME,
    interval_seconds=60.0,
    revision="RA",
    revision_comments="RA: Planned flight track - pre-flight",
)

print(f"ICARTT file: {os.path.basename(ict_path)}")
print(f"File size: {os.path.getsize(ict_path):,} bytes\n")

# Show header and first few data lines
with open(ict_path) as f:
    lines = f.readlines()

n_header = int(lines[0].split(",")[0])
print(f"Header: {n_header} lines")
print(f"Data: {len(lines) - n_header} points at 60-second intervals\n")

print("--- Header ---")
for line in lines[:n_header]:
    print(line, end="")

print("\n--- First 5 data lines ---")
for line in lines[n_header:n_header + 5]:
    print(line, end="")

ICARTT file: HYPLAN-DEMO-Flt-plan_B200_20250615_RA.ict
File size: 3,133 bytes

Header: 38 lines
Data: 32 points at 60-second intervals

--- Header ---
38, 1001
R. Pavlick
NASA JPL
Planned flight track - hyplan
HYPLAN-DEMO
1, 1
2025, 6, 15, 2026, 4, 14
0
Start_UTC, seconds, seconds from midnight UTC
6
1, 1, 1, 1, 1, 1
-9999, -9999, -9999, -9999, -9999, -9999
Latitude, Degrees, North positive, geodetic latitude
Longitude, Degrees, East positive, geodetic longitude
Altitude, meters, above sea level
speed, meters per second, ground speed
Bearing, degrees, degrees from north
SZA, degrees, solar zenith angle
0
17
PI_CONTACT_INFO: R. Pavlick
PLATFORM: King Air 200
LOCATION: N/A
ASSOCIATED_DATA: N/A
INSTRUMENT_INFO: Planned flight track generated by hyplan
DATA_INFO: Linearly interpolated planned flight track
UNCERTAINTY: N/A - planned track
ULOD_FLAG: -7777
ULOD_VALUE: N/A
LLOD_FLAG: -8888
LLOD_VALUE: N/A
DM_CONTACT_INFO: R. Pavlick
PROJECT_INFO: HYPLAN-DEMO
STIPULATIONS_ON_USE: Use as-is for

## 9. KML/KMZ for Google Earth

KML with waypoint markers and a flight path line. Both `.kml` and
`.kmz` files are generated. The `altitude_exaggeration` parameter
scales altitude in the 3D view (default 1.0 = true scale;
MovingLines uses 10).

In [11]:
kml_path = os.path.join(OUT_DIR, "demo_flight_plan.kml")
to_kml(
    plan, kml_path,
    takeoff_time=TAKEOFF_TIME,
    altitude_exaggeration=1.0,
)

print(f"KML: {kml_path}")
print(f"KMZ: {kml_path.replace('.kml', '.kmz')}")
print(f"KML size: {os.path.getsize(kml_path):,} bytes")
print(f"KMZ size: {os.path.getsize(kml_path.replace('.kml', '.kmz')):,} bytes")

# Show a snippet of the KML
with open(kml_path) as f:
    content = f.read()
# Show first Placemark
start = content.find("<Placemark>")
end = content.find("</Placemark>") + len("</Placemark>")
print(f"\n--- First waypoint Placemark ---")
print(content[start:end])

KML: /var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/hyplan_exports_naxevlsj/demo_flight_plan.kml
KMZ: /var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/hyplan_exports_naxevlsj/demo_flight_plan.kmz
KML size: 4,296 bytes
KMZ size: 930 bytes

--- First waypoint Placemark ---



## 10. GPX

Standard GPS exchange format. Creates a route with waypoints that
can be loaded into GPS devices and mapping applications.

In [12]:
gpx_path = os.path.join(OUT_DIR, "demo_flight_plan.gpx")
to_gpx(
    plan, gpx_path,
    mission_name=MISSION_NAME,
    takeoff_time=TAKEOFF_TIME,
)

print(f"GPX: {gpx_path}")
print(f"File size: {os.path.getsize(gpx_path):,} bytes\n")

with open(gpx_path) as f:
    print(f.read())

GPX: /var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/hyplan_exports_naxevlsj/demo_flight_plan.gpx
File size: 1,477 bytes

<?xml version="1.0" encoding="UTF-8"?>
<gpx xmlns="http://www.topografix.com/GPX/1/1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.topografix.com/GPX/1/1 http://www.topografix.com/GPX/1/1/gpx.xsd" version="1.1" creator="gpx.py -- https://github.com/tkrajina/gpxpy">
  <rte>
    <name>HYPLAN-DEMO</name>
    <rtept lat="34.910781" lon="-117.886391">
      <ele>704.6975999999997</ele>
      <time>2025-06-15T15:00:00</time>
      <name>WP#0</name>
      <cmt>takeoff</cmt>
    </rtept>
    <rtept lat="34.7" lon="-118.2">
      <ele>3047.999999999999</ele>
      <time>2025-06-15T15:06:22.479015</time>
      <name>WP#1</name>
      <cmt>climb</cmt>
    </rtept>
    <rtept lat="34.89999999999999" lon="-118.00000000000001">
      <ele>6095.999999999998</ele>
      <time>2025-06-15T15:11:25.465343</time>
      <name>WP#2</name>
      <c

## 11. Plain Text

Tab-separated text file matching the MovingLines `save2txt` format.
Note: longitude comes before latitude (opposite of the Excel format)
for MovingLines compatibility.

In [13]:
txt_path = os.path.join(OUT_DIR, "demo_flight_plan.txt")
to_txt(plan, txt_path, takeoff_time=TAKEOFF_TIME)

print(f"TXT: {txt_path}\n")
with open(txt_path) as f:
    print(f.read())

TXT: /var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/hyplan_exports_naxevlsj/demo_flight_plan.txt

#WP  Lon[+-180]  Lat[+-90]  Speed[m/s]  delayT[min]  Altitude[m]  CumLegT[H]  UTC[H]  LocalT[H]  LegT[H]  Dist[km]  CumDist[km]  Dist[Nmi]  CumDist[Nmi]  Speed[kt]  Altitude[kft]  SZA[deg]  AZI[deg]  Bearing[deg]  Climbt[min]  Comments  WPnames
0   -117.88639100  +34.91078100  104.73  0    704.7  0.00  15.00  7.14  0.11  40.1   0.0    21.6   0.0    203.6  2.31  -9999.0  -9999.0  0.0  0    takeoff  H1500
1   -118.20000000  +34.70000000  114.28  0    3048.0  0.11  15.11  7.23  0.08  34.6   40.1   18.7   21.6   222.1  10.00  -9999.0  -9999.0  0.0  0    climb  H1501
2   -118.00000000  +34.90000000  109.50  0    6096.0  0.19  15.19  7.32  0.09  33.7   74.7   18.2   40.3   212.9  20.00  -9999.0  -9999.0  45.0  0    descent  H1502
3   -118.20000000  +35.10000000  112.37  0    4572.0  0.28  15.28  7.40  0.10  38.4   108.4  20.8   58.5   218.4  15.00  -9999.0  -9999.0  180.0  0    climb  H1503
4 

## 12. Full Working Excel

The comprehensive 24-column format (A–X) that MovingLines uses as its
internal working format. This enables round-tripping: a MovingLines
user can open a hyplan-generated file and continue editing, or vice
versa.

Columns include both metric and imperial units, solar geometry (SZA, AZI),
cumulative distances, and timing information.

In [14]:
excel_path = os.path.join(OUT_DIR, "demo_full.xlsx")
to_excel(
    plan, excel_path,
    aircraft=b200,
    takeoff_time=TAKEOFF_TIME,
    mission_name=MISSION_NAME,
)

print(f"Full Excel: {excel_path}")
print(f"File size: {os.path.getsize(excel_path):,} bytes\n")

# Read it back and show all 24 columns
df_full = pd.read_excel(excel_path)
print(f"Columns ({len(df_full.columns)}): {list(df_full.columns)}\n")
df_full

Full Excel: /var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/hyplan_exports_naxevlsj/demo_full.xlsx
File size: 7,190 bytes

Columns (28): ['WP', 'Lat [+-90]', 'Lon [+-180]', 'Speed [m/s]', 'delayT [min]', 'Altitude [m]', 'CumLegT [hh:mm]', 'UTC [hh:mm]', 'LocalT [hh:mm]', 'LegT [hh:mm]', 'Dist [km]', 'CumDist [km]', 'Dist [Nmi]', 'CumDist [Nmi]', 'Speed [kt]', 'Altitude [kft]', 'SZA [deg]', 'AZI [deg]', 'Bearing [deg]', 'ClimbT [min]', 'Comments', 'WP names', 'HdWind [kt]', 'HdWind [m/s]', '2025-06-15', 'HYPLAN-DEMO', 'Created with', 'hyplan']



,WP,Lat [+-90],Lon [+-180],Speed [m/s],delayT [min],Altitude [m],CumLegT [hh:mm],UTC [hh:mm],LocalT [hh:mm],LegT [hh:mm],...,Bearing [deg],ClimbT [min],Comments,WP names,HdWind [kt],HdWind [m/s],2025-06-15,HYPLAN-DEMO,Created with,hyplan
0,0,34.910781,-117.886391,104.726190,0,704.6976,00:00:00,15:00:00,07:08:27.266000,00:06:22.479000,...,0,0,takeoff,H1500,0,0,NaN,NaN,NaN,NaN
1,1,34.700000,-118.200000,114.280159,0,3048.0000,00:06:22.479000,15:06:22.479000,07:13:34.479000,00:05:02.986000,...,0,0,climb,H1501,0,0,NaN,NaN,NaN,NaN
2,2,34.900000,-118.000000,109.503175,0,6096.0000,00:11:25.465000,15:11:25.465000,07:19:25.465000,00:05:07.956000,...,45,0,descent,H1502,0,0,NaN,NaN,NaN,NaN
3,3,35.100000,-118.200000,112.369365,0,4572.0000,00:16:33.421000,15:16:33.421000,07:23:45.421000,00:05:42.116000,...,180,0,climb,H1503,0,0,NaN,NaN,NaN,NaN
4,4,34.800000,-118.400000,97.381100,0,5486.4000,00:22:15.537000,15:22:15.537000,07:28:39.537000,00:10:31.494000,...,270,0,descent,H1504,0,0,NaN,NaN,NaN,NaN
5,5,34.910781,-117.886391,0.000000,0,704.6976,00:32:47.031000,15:32:47.031000,07:41:14.297000,00:00:00,...,0,0,NaN,H1505,0,0,NaN,NaN,NaN,NaN


## 13. Export All at Once

In practice, you'll often want to generate all formats for a single
flight plan. Here's a helper pattern that writes everything to a
named directory.

In [15]:
def export_all(plan, output_dir, aircraft=None, takeoff_time=None,
               mission_name="", pi_name="", institution=""):
    """Generate all export formats for a flight plan."""
    os.makedirs(output_dir, exist_ok=True)
    base = os.path.join(output_dir, mission_name or "flight_plan")
    date_str = takeoff_time.strftime("%Y%m%d") if takeoff_time else "undated"
    platform = getattr(aircraft, "aircraft_type", "unknown").replace(" ", "_")

    files = {}
    files["excel"] = f"{base}.xlsx"
    files["pilot_excel"] = f"{base}_for_pilots.xlsx"
    files["foreflight"] = f"{base}_FOREFLIGHT.csv"
    files["honeywell"] = f"{base}_Honeywell.csv"
    files["er2"] = f"{base}_ER2.csv"
    files["icartt"] = f"{base}-Flt-plan_{platform}_{date_str}_RA.ict"
    files["kml"] = f"{base}.kml"
    files["gpx"] = f"{base}.gpx"
    files["txt"] = f"{base}.txt"

    to_excel(plan, files["excel"], aircraft=aircraft,
             takeoff_time=takeoff_time, mission_name=mission_name)
    to_pilot_excel(plan, files["pilot_excel"], aircraft=aircraft,
                   takeoff_time=takeoff_time, mission_name=mission_name)
    to_foreflight_csv(plan, files["foreflight"], takeoff_time=takeoff_time)
    to_honeywell_fms(plan, files["honeywell"], takeoff_time=takeoff_time)
    to_er2_csv(plan, files["er2"], takeoff_time=takeoff_time)
    to_icartt(plan, files["icartt"], pi_name=pi_name,
              institution=institution, mission_name=mission_name,
              flight_date=takeoff_time.date() if takeoff_time else None,
              aircraft=aircraft, takeoff_time=takeoff_time)
    to_kml(plan, files["kml"], takeoff_time=takeoff_time)
    to_gpx(plan, files["gpx"], mission_name=mission_name,
           takeoff_time=takeoff_time)
    to_txt(plan, files["txt"], takeoff_time=takeoff_time)

    return files


# Run it
all_dir = os.path.join(OUT_DIR, "all_exports")
files = export_all(
    plan, all_dir,
    aircraft=b200,
    takeoff_time=TAKEOFF_TIME,
    mission_name=MISSION_NAME,
    pi_name="R. Pavlick",
    institution="NASA HQ",
)

print(f"All files written to: {all_dir}\n")
for fmt, path in files.items():
    size = os.path.getsize(path)
    print(f"  {fmt:<15s} {os.path.basename(path):<50s} {size:>8,} bytes")

All files written to: /var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/hyplan_exports_naxevlsj/all_exports

  excel           HYPLAN-DEMO.xlsx                                      7,190 bytes
  pilot_excel     HYPLAN-DEMO_for_pilots.xlsx                           6,279 bytes
  foreflight      HYPLAN-DEMO_FOREFLIGHT.csv                              394 bytes
  honeywell       HYPLAN-DEMO_Honeywell.csv                               310 bytes
  er2             HYPLAN-DEMO_ER2.csv                                     437 bytes
  icartt          HYPLAN-DEMO-Flt-plan_King_Air_200_20250615_RA.ict     3,132 bytes
  kml             HYPLAN-DEMO.kml                                       4,304 bytes
  gpx             HYPLAN-DEMO.gpx                                       1,477 bytes
  txt             HYPLAN-DEMO.txt                                       1,214 bytes


## Summary

| Function | Output | Compatible with |
|----------|--------|-----------------|
| `to_excel()` | Full 24-column `.xlsx` | MovingLines `write_to_excel()` |
| `to_pilot_excel()` | Simplified pilot `.xlsx` | MovingLines `save2xl_for_pilots_xlswriter()` |
| `to_foreflight_csv()` | ForeFlight CSV + oneline `.txt` | MovingLines ForeFlight export |
| `to_honeywell_fms()` | Honeywell FMS `.csv` | MovingLines Honeywell export |
| `to_er2_csv()` | ER-2 `.csv` | MovingLines ER-2 export |
| `to_icartt()` | ICARTT v1001 `.ict` | MovingLines `write_ict()` |
| `to_kml()` | `.kml` + `.kmz` | MovingLines `save2kml()` |
| `to_gpx()` | GPX route `.gpx` | MovingLines `save2gpx()` |
| `to_txt()` | Plain text `.txt` | MovingLines `save2txt()` |

| Helper | Purpose |
|--------|---------|
| `extract_waypoints(plan)` | Convert segment GeoDataFrame to waypoint table |
| `generate_wp_names(n, prefix, date)` | 5-char MovingLines-compatible waypoint names |
| `magnetic_declination(lat, lon)` | WMM magnetic declination in degrees |
| `true_to_magnetic(heading, dec)` | True heading → magnetic heading |
| `dd_to_ddm(lat, lon)` | Decimal degrees → `DD MM.MM` |
| `dd_to_ddms(lat, lon)` | Decimal degrees → `DD MM SS.S` |
| `dd_to_nddmm(lat, lon)` | Decimal degrees → `N37 24.21` / `W122 03.45` |
| `dd_to_foreflight_oneline(lat, lon)` | Decimal degrees → `N3724.210/W12203.450` |